# E04 — Thermal Expansion & Constrained Thermal Stress
*Exam tool — simplified 3-part structure.*

---

## Part 1 — Theory Recap

### Free (Unconstrained) Expansion
$$\Delta L = L \cdot \alpha \cdot \Delta T$$
The linear CTE $\alpha$ is assumed constant over the temperature range.

### Fully Constrained Thermal Stress
$$\sigma_{\text{thermal}} = E \cdot \alpha \cdot \Delta T$$
Derived from Hooke's law: the prevented strain $\varepsilon_{\text{prevented}} = \alpha \Delta T$ causes stress. The rod is in **compression** when $\Delta T > 0$ (expansion blocked).

### Partial Constraint (Gap Available)
$$F = E \cdot A_{\text{cross}} \cdot \left(\alpha \Delta T - \frac{\delta_{\text{gap}}}{L}\right)$$
If the gap $\delta_{\text{gap}} \geq \Delta L$, the expansion is fully absorbed and $F = 0$.

### Constrained Force (Circular Cross-Section)
$$A_{\text{cross}} = \frac{\pi d^2}{4}$$

### Critical Temperature Rise (Yielding)
$$\Delta T_{\text{crit}} = \frac{\sigma_{\text{yield}}}{E \cdot \alpha}$$
At $\Delta T > \Delta T_{\text{crit}}$ the thermal stress exceeds yield — the part deforms permanently.

### Combined with Press-Fit
When a polymer hub on a steel shaft heats up, the hub expands more than the shaft ($\alpha_{\text{polymer}} > \alpha_{\text{steel}}$), **relieving** the interference. Use:
$$\delta_{\text{effective}} = \delta_0 - (\alpha_{\text{polymer}} - \alpha_{\text{steel}}) \cdot D_1 \cdot \Delta T$$

### Parameter Table
| Symbol | Description | Unit |
|--------|-------------|------|
| $L$ | Rod/component length | m |
| $d$ | Diameter (circular cross-section) | m |
| $\alpha$ | Linear CTE | 1/°C |
| $\Delta T$ | Temperature change ($>0$ = heating) | °C |
| $E$ | Tensile modulus | Pa |
| $\sigma_{\text{yield}}$ | Yield stress | Pa |
| $\delta_{\text{gap}}$ | Available clearance gap | m |

---

## ⚠ Common Exam Pitfalls
1. **Wrong modulus** — use the **tensile** modulus, not the flexural modulus, for thermal stress.
2. **Cross-section area** — compute $A = \pi d^2/4$ before calculating force; don't confuse with area moment.
3. **Sign convention** — constrained expansion creates **compressive** stress in the rod. Report magnitude for yield comparison.
4. **Gap direction** — if the gap absorbs all expansion ($\delta_{\text{gap}} \geq \Delta L$), the force is exactly zero.

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Problem Inputs  (edit values here)
# ═══════════════════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '..')
import numpy as np

from utils.unit_registry import ureg, Q_, strip_units
from utils.material_db import THERMAL_EXPANSION

# ── Material ─────────────────────────────────────────────────────────────────
MAT_KEY = 'PP'   # keys: 'PP', 'POM', 'PC', 'PA66', 'PVC'
mat_cte = THERMAL_EXPANSION.get(MAT_KEY, {})
CTE     = mat_cte.get('CTE', 90e-6)   # fallback 90e-6 1/°C if key missing
CTE_src = mat_cte.get('description', 'fallback value — add to THERMAL_EXPANSION')

# ── Geometry ─────────────────────────────────────────────────────────────────
L_rod      = Q_(200.0, 'mm')    # rod length
d_rod      = Q_(  8.0, 'mm')    # rod diameter (circular cross-section)
delta_gap  = Q_(  0.0, 'mm')    # available expansion gap (0 = fully constrained)

# ── Thermal loading ───────────────────────────────────────────────────────────
delta_T = 40.0   # [°C] temperature change (positive = heating)

# ── Material properties ───────────────────────────────────────────────────────
E_tensile   = Q_(1300.0, 'MPa')   # tensile modulus (NOT flexural)
sigma_yield = Q_(  30.0, 'MPa')   # yield stress (from datasheet)

print(f'CTE source   : {CTE_src}')
print(f'CTE [1/°C]   : {CTE:.2e}')


CTE source   : PP homopolymer, 23°C, Borealis datasheet
CTE [1/°C]   : 9.00e-05


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Functions
# ═══════════════════════════════════════════════════════════════════════════════

def free_expansion(L_m, alpha, dT):
    """Unconstrained thermal expansion ΔL = L·α·ΔT. Returns metres."""
    return L_m * alpha * dT


def constrained_stress(E_Pa, alpha, dT):
    """Fully constrained thermal stress σ = E·α·ΔT. Returns Pa (magnitude)."""
    return E_Pa * alpha * abs(dT)


def constraint_force(E_Pa, A_m2, alpha, dT, delta_gap_m, L_m):
    """Thermal constraint force with optional gap.
    F = E·A·(α·ΔT - δ_gap/L). Clamped to zero if gap absorbs all expansion."""
    net_strain = alpha * dT - delta_gap_m / L_m
    if net_strain <= 0:
        print('  NOTE: gap >= free expansion — no constraint force develops (F = 0)')
        return 0.0
    return E_Pa * A_m2 * net_strain


print('Functions defined.')


Functions defined.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Execution
# ═══════════════════════════════════════════════════════════════════════════════

L_m       = strip_units(L_rod.to('m'))
d_m       = strip_units(d_rod.to('m'))
gap_m     = strip_units(delta_gap.to('m'))
E_Pa      = strip_units(E_tensile.to('Pa'))
sy_Pa     = strip_units(sigma_yield.to('Pa'))
A_cross   = np.pi * d_m**2 / 4.0

direction = 'heating' if delta_T > 0 else 'cooling'

# --- Step 1: Material and geometry -------------------------------------------
print('--- Step 1: Material & Geometry ---')
print(f"  {'Material':<30}: {MAT_KEY}")
print(f"  {'CTE α [μm/(m·°C)]':<30}: {CTE*1e6:.1f}")
print(f"  {'L [mm]':<30}: {L_m*1e3:.1f}")
print(f"  {'d [mm]':<30}: {d_m*1e3:.1f}")
print(f"  {'A_cross [mm²]':<30}: {A_cross*1e6:.3f}")
print(f"  {'ΔT [°C]':<30}: {delta_T:+.1f}  ({direction})")
print(f"  {'Gap δ_gap [mm]':<30}: {gap_m*1e3:.3f}")

# --- Step 2: Free expansion --------------------------------------------------
dL_free = free_expansion(L_m, CTE, delta_T)
print()
print('--- Step 2: Free Expansion ---')
print(f"  {'ΔL_free = L·α·ΔT [mm]':<30}: {dL_free*1e3:.4f}")
print(f"  {'ΔL_free / L [%]':<30}: {dL_free/L_m*100:.4f} %")

# --- Step 3: Constrained stress and force ------------------------------------
sigma_th  = constrained_stress(E_Pa, CTE, delta_T)
F_constr  = constraint_force(E_Pa, A_cross, CTE, delta_T, gap_m, L_m)
SF_yield  = sy_Pa / sigma_th if sigma_th > 0 else float('inf')
dT_crit   = sy_Pa / (E_Pa * CTE)

print()
print('--- Step 3: Constrained Stress & Force ---')
print(f"  {'Gap absorbed [mm]':<30}: {min(gap_m, dL_free)*1e3:.4f}")
print(f"  {'Net prevented expansion [mm]':<30}: {max(dL_free - gap_m, 0)*1e3:.4f}")
print(f"  {'σ_thermal [MPa] (fully constrained)':<30}: {sigma_th/1e6:.3f}  (compressive when ΔT>0)")
print(f"  {'F_constraint [N]':<30}: {F_constr:.1f}")
print(f"  {'σ_yield [MPa]':<30}: {sy_Pa/1e6:.1f}")
print(f"  {'Safety factor (yield/σ_th)':<30}: {SF_yield:.2f}")
print()
print(f"  {'ΔT_crit (yielding at) [°C]':<30}: {dT_crit:.1f}")
print(f"  {'Current ΔT / ΔT_crit':<30}: {abs(delta_T)/dT_crit:.2f}  ({abs(delta_T)/dT_crit*100:.1f}% of yield onset)")


--- Step 1: Material & Geometry ---
  Material                      : PP
  CTE α [μm/(m·°C)]             : 90.0
  L [mm]                        : 200.0
  d [mm]                        : 8.0
  A_cross [mm²]                 : 50.265
  ΔT [°C]                       : +40.0  (heating)
  Gap δ_gap [mm]                : 0.000

--- Step 2: Free Expansion ---
  ΔL_free = L·α·ΔT [mm]         : 0.7200
  ΔL_free / L [%]               : 0.3600 %

--- Step 3: Constrained Stress & Force ---
  Gap absorbed [mm]             : 0.0000
  Net prevented expansion [mm]  : 0.7200
  σ_thermal [MPa] (fully constrained): 4.680  (compressive when ΔT>0)
  F_constraint [N]              : 235.2
  σ_yield [MPa]                 : 30.0
  Safety factor (yield/σ_th)    : 6.41

  ΔT_crit (yielding at) [°C]    : 256.4
  Current ΔT / ΔT_crit          : 0.16  (15.6% of yield onset)


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Validation
# ═══════════════════════════════════════════════════════════════════════════════

pass_stress = sigma_th <= sy_Pa
overall     = pass_stress

print('--- VALIDATION ---')
print(f"  Thermal stress  : {sigma_th/1e6:.3f} MPa  <=  {sy_Pa/1e6:.1f} MPa (yield)"
      f"  |  {'PASS' if pass_stress else 'FAIL'}")
if gap_m >= dL_free:
    print(f"  Gap check       : gap ({gap_m*1e3:.3f} mm) >= ΔL ({dL_free*1e3:.4f} mm)  |  no stress")
else:
    print(f"  Gap check       : gap ({gap_m*1e3:.3f} mm) < ΔL ({dL_free*1e3:.4f} mm)  |  partial constraint")
print(f"  OVERALL         : {'PASS' if overall else 'FAIL'}")


--- VALIDATION ---
  Thermal stress  : 4.680 MPa  <=  30.0 MPa (yield)  |  PASS
  Gap check       : gap (0.000 mm) < ΔL (0.7200 mm)  |  partial constraint
  OVERALL         : PASS
